In [0]:
%sql
-- Standard calendar dimension: 1940-01-01 to 2030-12-31
CREATE OR REPLACE TABLE adb_caresync.silver.calendar
COMMENT 'Standard calendar dimension from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS d
)
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT)              AS date_key,
  d                                                    AS date,
  year(d)                                              AS year,
  CASE WHEN month(d) <= 6 THEN 1 ELSE 2 END            AS half_year,
  quarter(d)                                           AS quarter,
  concat('Q', quarter(d))                              AS quarter_name,
  concat(year(d), '-Q', quarter(d))                    AS year_quarter,
  month(d)                                             AS month,
  date_format(d, 'MMMM')                               AS month_name,
  date_format(d, 'MMM')                                AS month_short_name,
  date_format(d, 'yyyy-MM')                            AS year_month,
  CAST(date_format(d, 'yyyyMM') AS INT)                AS year_month_key,
  weekofyear(d)                                        AS week_of_year_iso,
  extract(YEAROFWEEK FROM d)                           AS iso_year,
  CAST(ceil(dayofmonth(d) / 7.0) AS INT)               AS week_of_month,
  dayofmonth(d)                                        AS day_of_month,
  dayofyear(d)                                         AS day_of_year,
  dayofweek(d)                                         AS day_of_week,       -- 1 = Sunday ... 7 = Saturday
  extract(DAYOFWEEK_ISO FROM d)                        AS day_of_week_iso,   -- 1 = Monday ... 7 = Sunday
  date_format(d, 'EEEE')                               AS day_name,
  date_format(d, 'EEE')                                AS day_short_name,
  dayofweek(d) IN (1, 7)                               AS is_weekend,
  dayofweek(d) NOT IN (1, 7)                           AS is_weekday,
  date_trunc('WEEK', d)::DATE                          AS week_start_date,   -- Monday
  date_add(date_trunc('WEEK', d)::DATE, 6)             AS week_end_date,     -- Sunday
  trunc(d, 'MM')                                       AS month_start_date,
  last_day(d)                                          AS month_end_date,
  dayofmonth(last_day(d))                              AS days_in_month,
  date_trunc('QUARTER', d)::DATE                       AS quarter_start_date,
  date_add(add_months(date_trunc('QUARTER', d)::DATE, 3), -1) AS quarter_end_date,
  trunc(d, 'YYYY')                                     AS year_start_date,
  make_date(year(d), 12, 31)                           AS year_end_date,
  d = trunc(d, 'MM')                                   AS is_month_start,
  d = last_day(d)                                      AS is_month_end,
  d = date_trunc('QUARTER', d)::DATE                   AS is_quarter_start,
  d = date_add(add_months(date_trunc('QUARTER', d)::DATE, 3), -1) AS is_quarter_end,
  d = trunc(d, 'YYYY')                                 AS is_year_start,
  d = make_date(year(d), 12, 31)                       AS is_year_end,
  (year(d) % 4 = 0 AND year(d) % 100 <> 0) OR year(d) % 400 = 0 AS is_leap_year,
  CAST(floor(year(d) / 10) * 10 AS INT)                AS decade
FROM dates
ORDER BY d;

In [0]:
%sql
SELECT count(*) AS total_rows,
       count(DISTINCT date_key) AS distinct_keys,
       min(date) AS min_date,
       max(date) AS max_date,
       count(DISTINCT year) AS years
FROM adb_caresync.silver.calendar;

In [0]:
%sql
SELECT * FROM adb_caresync.silver.calendar
WHERE date BETWEEN DATE'2024-12-28' AND DATE'2025-01-05'
ORDER BY date;